In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import tensorflow.keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

pi = 3.14159265359

maxval=1e9
minval=1e-9

2025-11-17 18:28:54.811805: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-17 18:28:55.778269: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import *
from models.mlp_encoder_model_nonquantized import *
from AnnealingScheduler import *

In [3]:
seed = 10
tf.random.set_seed(seed)
random.seed(seed)

##### Checklist:
* no constraint -> done
* gaussian noise -> in progress
* min80e -> not started
* min400e, max8000e -> not started

In [4]:
#dataset_base_dir = "/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/"
dataset_base_dir = "/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train_contained")
dataset_validation_dir = os.path.join(dataset_base_dir, "test_contained")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train_contained_slim")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val_contained_slim")

dirs_to_create = [
    tfrecords_dir_train,
    tfrecords_dir_val,
    dataset_train_dir,
    dataset_validation_dir
]

# Create each directory if it doesn't exist
for directory in dirs_to_create:
    os.makedirs(directory, exist_ok=True)

In [5]:
print(f'Number of training files: {len(os.listdir(dataset_train_dir))}')
print(f'Number of validation files: {len(os.listdir(dataset_validation_dir))}')

Number of training files: 80
Number of validation files: 20


In [6]:
batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_validation_dir))

In [7]:
start_time = time.time()
validation_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_validation_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = val_batch_size,
    file_count = val_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshold = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, 
    files_from_end = True,

    tfrecords_dir = tfrecords_dir_val,
    use_time_stamps = [0,19],
    max_workers = 1,
    #load_from_tfrecords_dir = tfrecords_dir_val
)

print("--- Validation generator %s seconds ---" % (time.time() - start_time))

Processing Files...: 100%|██████████| 20/20 [00:05<00:00,  3.76it/s]


Directory /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_slim is removed...


Saving batches as TFRecords: 100%|██████████| 24/24 [00:06<00:00,  3.50it/s]


Metadata saved successfully ast /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_slim/metadata.json
Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_slim/metadata.json
--- Validation generator 12.506949424743652 seconds ---


In [8]:
# training generator
start_time = time.time()
training_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_train_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = batch_size,
    file_count = train_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshold = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, # True 

    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = [0,19],
    max_workers = 1,
    #load_from_tfrecords_dir = tfrecords_dir_train
)
print("--- Training generator %s seconds ---" % (time.time() - start_time))

Processing Files...: 100%|██████████| 80/80 [00:20<00:00,  3.86it/s]


Directory /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_slim is removed...


Saving batches as TFRecords: 100%|██████████| 95/95 [00:26<00:00,  3.65it/s]


Metadata saved successfully ast /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_slim/metadata.json
Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_slim/metadata.json
--- Training generator 47.13702321052551 seconds ---


In [9]:
training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle = True,
    seed = seed,
    quantize = False
)

validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_val,
    shuffle = True,
    seed = seed,
    quantize = False
)


Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_slim/metadata.json


Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_slim/metadata.json


In [10]:
model=CreateModel_Slim_SoftQuantizer((16,16,2), initial_thresholds=[247.8, 668.4, 1662.9], threshold_offset=80.0, initial_levels=np.array([0.0, 1.0, 2.0, 3.0]))
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_sse_loss
)

model.summary()

Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 soft_quantizer_output (Sof  (None, 16, 16, 2)            4         ['input_pxls[0][0]']          
 tQuantizeLayer)                                                                                  
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['soft_quantizer_output[0][0]'
 Pooling2D)                                                         ]                             
                                                                                 

In [11]:
# training
pitch = '50x12P5'
fingerprint = '%08x' % random.randrange(16**8)
base_dir = '/data/dajiang/smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_-8_8_offsets_weights/'
weights_dir = base_dir + 'weights-{}-bs{}-{}-2t-mlp_SLIM-offsets-soft_quantizer-checkpoints'.format(pitch, batch_size, fingerprint)

# create output directories
if os.path.isdir(base_dir):
    os.mkdir(weights_dir)
else:
    os.mkdir(base_dir)
    os.mkdir(weights_dir)
    
checkpoint_filepath = weights_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'
mcp = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=False,
)

print('Model fingerprint: {}'.format(fingerprint))

Model fingerprint: 34c2da80


In [12]:
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=67.0, 
    verbose=1      
)

In [ ]:
history = model.fit(x=training_generator,
                    validation_data=validation_generator,
                    callbacks=[mcp, scheduler_callback],
                    epochs=1000,
                    shuffle=False, # shuffling now occurs within the data-loader
                    verbose=1)


Epoch 1: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 1/1000


2025-11-17 18:30:48.786886: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8906
2025-11-17 18:30:48.792332: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-11-17 18:30:48.870773: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7fa36a071220 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-17 18:30:48.870848: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB MIG 1g.5gb, Compute Capability 8.0
2025-11-17 18:30:48.880445: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:255] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-17 18:30:48.944377: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-11-17 18:30

95/95 [==============================] - 17s 145ms/step - loss: 755.5140 - val_loss: 533.7734

Epoch 2: Annealing 'k' set to 1.0002
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 2/1000
95/95 [==============================] - 13s 140ms/step - loss: 474.7922 - val_loss: 409.4509

Epoch 3: Annealing 'k' set to 1.0007
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 3/1000
95/95 [==============================] - 13s 137ms/step - loss: 346.6293 - val_loss: 297.2009

Epoch 4: Annealing 'k' set to 1.0015
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 4/1000
95/95 [==============================] - 13s 131ms/step - loss: 276.2319 - val_loss: 258.4821

Epoch 5: Annealing 'k' set to 1.0026
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 5/1000
95/95 [==============================] - 13s 132ms/step - loss: 248.5794 - val_loss: 237.9132

Epoch 6: Annealing 'k' set to 1.0041
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 6/1000
95/95 [==============================] - 13s 134ms/step - loss: 231.0379 - 